# Generate Mock Data
To anonymise any personal information, a simulated dataset is used in this work.

Some simple trends are implemented, including a steadily increase cost of rent.

In [ ]:
from datetime import datetime, timedelta
import random
import pandas as pd

def calculate_salary(balance, transactions, current_date):
    salary = round(random.uniform(2300, 2800), 2)
    balance += salary
    transactions.append([
        current_date.strftime("%d/%m/%Y"),
        "SALARY PAYMENT ACME ENGINEERING LTD",
        "Salary",
        "Income",
        f"{salary:,.2f}",
        "",
        f"{balance:,.2f}"
    ])

    return balance, transactions

def calculate_interest(balance, transactions, current_date):
    interest = round(random.uniform(balance * 0.001, balance * 0.005), 2)
    balance += interest
    transactions.append([
        current_date.strftime("%d/%m/%Y"),
        "INTEREST PAYMENT",
        "Interest",
        "Income",
        f"{interest:,.2f}",
        "",
        f"{balance:,.2f}"
    ])

    return balance, transactions

def calculate_rent(rent, balance, transactions, current_date, last_increase_date):
    # Months since last rent increase
    months_since_increase = (current_date - last_increase_date).days // 30
    if months_since_increase >= 9:
        # Probability of rent increase ramping from 0% at 9 months to 100% at 15 months
        probability = min((months_since_increase - 9) / (15 -9), 1.0)
        if random.random() < probability:
            rent *= 1.03  # 3% increase
            rent = round(rent, 2)
            last_increase_date = current_date
            months_since_increase = 0

    balance -= rent
    transactions.append([
    current_date.strftime("%d/%m/%Y"),
    "DIRECT DEBIT TO LANDLORD RENT PAYMENT",
    "Rent",
    "Housing",
    "",
    f"{rent:,.2f}",
    f"{balance:,.2f}"
    ])

    return rent, balance, transactions, last_increase_date

def calculate_utilities(balance, transactions, current_date, vendors):
    utility = random.choice(vendors["Utilities"])
    amount = round(random.uniform(25, 95), 2)
    balance -= amount
    transactions.append([
        current_date.strftime("%d/%m/%Y"),
        f"DIRECT DEBIT TO {utility.upper()}",
        utility,
        "Utilities",
        "",
        f"{amount:,.2f}",
        f"{balance:,.2f}"
    ])

    return balance, transactions

def calculate_random_expenses(balance, transactions, current_date, vendors):
    # More transactions in November and December due to holiday shopping
    if current_date.strftime("%m") >= "11":
        num_tx = random.randint(0, 5)
    else:
        num_tx = random.randint(0, 3)

    for _ in range(num_tx):
        category = random.choice(list(vendors.keys()))
        if category in ["Housing", "Bills"]:
            continue

        vendor = random.choice(vendors[category])

        if category == "Groceries":
            amount = round(random.uniform(8, 65), 2)
        elif category == "Food & Drink":
            amount = round(random.uniform(3, 30), 2)
        elif category == "Transport":
            amount = round(random.uniform(8, 60), 2)
        elif category == "Shopping":
            amount = round(random.uniform(15, 180), 2)
        elif category == "Recreation":
            amount = round(random.uniform(10, 60), 2)
        elif category == "Fitness":
            amount = round(random.uniform(20, 80), 2)
        elif category == "Subscriptions":
            amount = round(random.uniform(2, 18), 2)
        else:
            amount = round(random.uniform(5, 50), 2)

        balance -= amount

        transactions.append([
            current_date.strftime("%d/%m/%Y"),
            f"CARD PAYMENT TO {vendor.upper()}",
            vendor,
            category,
            "",
            f"{amount:,.2f}",
            f"{balance:,.2f}"
        ])

    return balance, transactions

In [ ]:
from datetime import datetime, timedelta
import random
import pandas as pd

random.seed(42)

vendors = {
    "Groceries": ["Tesco", "Sainsburys", "Morrisons", "Aldi", "Lidl", "Asda"],
    "Food & Drink": ["Costa Coffee", "Starbucks", "Greggs", "Pret", "Uber Eats", "Deliveroo", "Nandos"],
    "Utilities": ["Octopus Energy", "Virgin Media", "United Utilities"],
    "Transport": ["Shell", "BP", "Trainline", "Uber"],
    "Recreation": ["Steam", "Cineworld", "Waterstones"],
    "Shopping": ["Amazon", "Currys", "Ikea", "JD Sports"],
    "Fitness": ["PureGym", "Decathlon"],
    "Subscriptions": ["Spotify", "Netflix", "Disney Plus", "Apple"],
    "Bills": ["Council Tax"],
    "Housing": ["Rent"]
}

start_date = datetime(2020, 1, 1)
end_date = datetime(2026, 12, 31)

transactions = []
balance = 900.00
rent_amount_start = 750.00
rent = 750.00
last_increase_date = start_date

current_date = start_date

while current_date <= end_date:
    # Salary and interest once a month
    if current_date.day == 1:
        balance, transactions = calculate_salary(balance, transactions, current_date)
        balance, transactions = calculate_interest(balance, transactions, current_date)

    # Rent
    if current_date.day == 2:
        rent, balance, transactions, last_increase_date = calculate_rent(rent, balance, transactions, current_date, last_increase_date)

    # Utilities monthly
    if current_date.day in [3, 5, 7]:
        balance, transactions = calculate_utilities(balance, transactions, current_date, vendors)

    # Random daily transactions
    balance, transactions = calculate_random_expenses(balance, transactions, current_date, vendors)

    current_date += timedelta(days=1)

# create DataFrame and export to Excel
df = pd.DataFrame(transactions, columns=[
    "Date",
    "Description",
    "Description - cleaned",
    "Category",
    "Money in",
    "Money Out",
    "Balance"
])

xlsx_path = "Mock data.xlsx"
try:
    df.to_excel(xlsx_path, index=False, sheet_name="Mock data")
except PermissionError:
    print("File already exists or is open")

print(f"Generated {len(df)} synthetic transactions.")

Create table of category and whether or not it is an essential expense.

In [ ]:
essential_map = {
    "Fitness": False,
    "Food & Drink": False,
    "Groceries": True,
    "Housing": True,
    "Income": True,
    "Recreation": False,
    "Shopping": False,
    "Subscriptions": False,
    "Transport": True,
    "Utilities": True
}
essential_df = pd.DataFrame({"Category": df["Category"].unique()})
essential_df["Essential"] = essential_df["Category"].map(essential_map)
essential_df

from pathlib import Path

if Path(xlsx_path).exists():
    mode = "a"
else:
    mode = "w"

with pd.ExcelWriter(
    xlsx_path,
    engine="openpyxl",
    mode=mode,
    if_sheet_exists="replace" if mode == "a" else None
) as writer:
    essential_df.to_excel(
        writer,
        sheet_name="Essential",
        index=False
    )